# Prueba individual · Sesión 03
### Career Tech Hub · Un caso de datos de principio a fin

**Nombre:** *(Daniel Ledesma)*

Trabajas con los mismos datos de la sesión. Lee antes el PDF de la prueba: ahí están las instrucciones y la rúbrica.

- Haz **al menos dos** de las partes A (pandas), B (SQL) y C (Excel o Power BI).
- La **parte D es obligatoria**.
- Este cuaderno tiene que estar en la misma carpeta que el cuaderno de la sesión, junto a la carpeta `datos`.

In [1]:
import sqlite3
import pandas as pd

# Las hojas del Excel (para la parte A)
dias = pd.read_excel("datos/electricidad_para_panel.xlsx", sheet_name="dias")
generacion = pd.read_excel("datos/electricidad_para_panel.xlsx", sheet_name="generacion")

# La base de datos (para la parte B)
conexion = sqlite3.connect("datos/electricidad.db")

def sql(consulta):
    # Lanza una consulta y devuelve el resultado como tabla de pandas.
    return pd.read_sql(consulta, conexion)

print("Hoja dias:", dias.shape[0], "filas ·", list(dias.columns))
sql("SELECT type AS tipo, name AS nombre FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name")

Hoja dias: 1415 filas · ['fecha', 'dia_semana', 'mes', 'festivo', 'descanso', 't_med', 'demanda_gwh', 'año']


,tipo,nombre
0,table,ciudades
1,table,demanda
2,table,festivos
3,table,generacion
4,table,predicciones
5,table,tiempo
6,view,tabla_dias
7,view,temperatura_nacional


Columnas de las tablas de la base de datos:

| Tabla | Columnas |
|---|---|
| `demanda` | `fecha`, `demanda_gwh` |
| `generacion` | `fecha`, `tecnologia`, `tipo`, `gwh` |
| `tiempo` | `fecha`, `ciudad`, `t_max`, `t_min`, `t_med`, `lluvia_mm`, `viento_kmh`, `sol_mj` |
| `ciudades` | `ciudad`, `latitud`, `longitud`, `habitantes` |
| `festivos` | `fecha`, `festivo` |
| `temperatura_nacional` (vista) | `fecha`, `t_med` |
| `tabla_dias` (vista) | `fecha`, `demanda_gwh`, `t_med`, `festivo`, `demanda_ayer`, `demanda_hace_7` |

# Parte A · pandas

Usa la tabla `dias`.

**A1.** ¿Cuántos días superaron los 800 GWh de demanda? ¿Cuántos fueron en cada año?

In [2]:
# Muestro las primeras 5 filas de las columnas "fecha" y "demanda_gwh", para comprobar cómo se llaman las columnas, en qué formato viene la fecha y que pinta tienen los números de demanda.
dias[["fecha", "demanda_gwh"]].head(5)

,fecha,demanda_gwh
0,2023-01-01,506.425906
1,2023-01-02,621.645824
2,2023-01-03,677.052137
3,2023-01-04,689.160950
4,2023-01-05,671.291045


In [3]:
# Quiero conseguir los días que superaron los 800GWh de demanda, y que me muestre la fecha y la demanda de esos días.
dias[["fecha", "demanda_gwh"]][dias["demanda_gwh"] > 800]

,fecha,demanda_gwh
23,2023-01-24,825.630213
24,2023-01-25,816.153242
25,2023-01-26,817.260311
26,2023-01-27,806.430869
30,2023-01-31,803.698920
...,...,...
1341,2026-09-03,827.262640
1342,2026-09-04,838.597950
1345,2026-09-07,826.275390
1346,2026-09-08,835.150550


In [4]:
# Ahora quiero contar cuántos días superaron los 800GWh de demanda.
print(dias[["fecha", "demanda_gwh"]][dias["demanda_gwh"] > 800].shape[0], "días superaron los 800GWh de demanda. ")

115 días superaron los 800GWh de demanda. 


In [5]:
# Filtro por años según la demanda superior a 800 GWh y cuento cuántos días por año superaron esa demanda.
print("Días por año que superaron los 800GWh de demanda:")
print(dias[["año"]][dias["demanda_gwh"] > 800].value_counts())

Días por año que superaron los 800GWh de demanda:
año 
2026    64
2025    25
2023    16
2024    10
Name: count, dtype: int64


**A2.** Haz una tabla con la demanda media de cada año, separando los días laborables de los días de descanso (columna `descanso`). ¿Qué porcentaje menos se gasta en un día de descanso, cada año?

In [6]:
# Compruebo qué días son laborables y cuáles son días de descanso.
dias["descanso"].value_counts()

descanso
False    982
True     433
Name: count, dtype: int64

In [7]:
# Demanda media por año y por tipo de día (laborable o descanso).
dias.groupby(["año", "descanso"])["demanda_gwh"].mean()

año   descanso
2023  False       707.720644
      True        605.501375
2024  False       710.062423
      True        612.015598
2025  False       732.488655
      True        633.228393
2026  False       749.862854
      True        643.033811
Name: demanda_gwh, dtype: float64

In [8]:
#Calculo la media de demanda por año para los días de descanso
media_dias_descanso = dias[dias["descanso"] == True].groupby("año")["demanda_gwh"].mean()
media_dias_descanso

año
2023    605.501375
2024    612.015598
2025    633.228393
2026    643.033811
Name: demanda_gwh, dtype: float64

In [9]:
#Calculo la media de demanda por año para los días laborales.
media_dias_laborables = dias[dias["descanso"] == False].groupby("año")["demanda_gwh"].mean()
media_dias_laborables

año
2023    707.720644
2024    710.062423
2025    732.488655
2026    749.862854
Name: demanda_gwh, dtype: float64

In [10]:
# Calculo el porcentaje de diferencia entre la media de los días laborables y la media de los días de descanso para cada año.
print("Porcentaje medio de gasto entre días laborables y días de descanso por año:")
porcentaje_menos_descanso = (media_dias_laborables - media_dias_descanso) / media_dias_laborables * 100
porcentaje_menos_descanso.round(2)

Porcentaje medio de gasto entre días laborables y días de descanso por año:


año
2023    14.44
2024    13.81
2025    13.55
2026    14.25
Name: demanda_gwh, dtype: float64

In [11]:
# Calculo el porcentaje máximo y mínimo de diferencia entre la media de los días laborables y la media de los días de descanso.
print("El porcentaje máximo es:", porcentaje_menos_descanso.max().round(2), "%")
print("El porcentaje mínimo es:", porcentaje_menos_descanso.min().round(2), "%")

El porcentaje máximo es: 14.44 %
El porcentaje mínimo es: 13.55 %


**A3.** ¿Cuál fue el mes con mayor demanda media de los cuatro años (di el mes y el año)? ¿Y el de menor?

In [12]:
# Calculo la demanda media por mes de cada año.
demanda_media_mes = dias.groupby(["año", "mes"])["demanda_gwh"].mean()
demanda_media_mes

año   mes
2023  1      714.000848
      2      734.294114
      3      665.797137
      4      610.557810
      5      620.753786
      6      665.779343
      7      735.470937
      8      704.179547
      9      658.758496
      10     644.841359
      11     670.439727
      12     687.173872
2024  1      721.082959
      2      701.224980
      3      668.214389
      4      643.243569
      5      630.364662
      6      654.915852
      7      734.732135
      8      724.448240
      9      665.037387
      10     656.158678
      11     663.618507
      12     699.181688
2025  1      739.948473
      2      723.724240
      3      705.734611
      4      626.127690
      5      630.762264
      6      728.752704
      7      759.066657
      8      719.857451
      9      694.962081
      10     661.085055
      11     704.312938
      12     733.594810
2026  1      778.773568
      2      736.125984
      3      693.055557
      4      632.591222
      5      648.062540
      

In [13]:
# Ordeno la demanda por mes de cada año de mayor a menor para ver cuál es el mes con mayor demanda media de todos los años.
print(demanda_media_mes.sort_values(ascending=False).head(1).round(2))
print("La demanda media más alta se da en el mes de Julio del año 2026, con una demanda de 814.83 GWh.")

año   mes
2026  7      814.83
Name: demanda_gwh, dtype: float64
La demanda media más alta se da en el mes de Julio del año 2026, con una demanda de 814.83 GWh.


In [14]:
# Ordeno la demanda por mes de cada año de menor a mayor para ver cuál es el mes con menor demanda media.
print(demanda_media_mes.sort_values(ascending=True).head(1).round(2))
print("La demanda media más baja se da en el mes de Abril del año 2023, con una demanda de 610.56 GWh.")


año   mes
2023  4      610.56
Name: demanda_gwh, dtype: float64
La demanda media más baja se da en el mes de Abril del año 2023, con una demanda de 610.56 GWh.


# Parte B · SQL

Usa la función `sql("...")` con la base de datos.

**B1.** Saca los cinco días más calurosos (vista `temperatura_nacional`) junto con su demanda. ¿Son también días de mucha demanda? Explica por qué en una o dos frases.

In [ ]:
# Selecciono fecha y temperatura para ordenarlos según 5 días más calurosos
sql("SELECT fecha, t_med FROM temperatura_nacional ORDER BY t_med DESC LIMIT 5")

,fecha,t_med
0,2025-08-16,31.81
1,2025-08-17,31.79
2,2023-08-23,31.61
3,2025-08-11,31.49
4,2025-06-29,31.43


In [ ]:
# Se unen dos tablas demanda y temperatura_nacional con el fin de conocer si los días más calurosos coinciden con la demanda más alta
sql("SELECT temperatura_nacional.fecha, t_med, demanda_gwh FROM demanda JOIN temperatura_nacional ON demanda.fecha = temperatura_nacional.fecha ORDER BY t_med DESC limit 5")

,fecha,t_med,demanda_gwh
0,2025-08-16,31.81,712.442720
1,2025-08-17,31.79,698.786118
2,2023-08-23,31.61,805.114493
3,2025-08-11,31.49,770.154701
4,2025-06-29,31.43,707.452281


De los cinco días más calurosos, solo dos superan la media de demanda de días laborables de su respectivo año, y los otros tres están cerca o por debajo, así que el calor extremo no garantiza una demanda alta. Podría deberse más a si es un día laborable o de descanso, que pesa más que la propia temperatura.

**B2.** Para cada año, calcula la generación solar fotovoltaica en TWh y qué porcentaje supone de toda la generación del año.

In [16]:
# Tu código aquí


**B3.** ¿Cuántos días de lluvia (1 mm o más) ha tenido cada ciudad? Ordena de más a menos.

In [17]:
# Tu código aquí


**Reto (opcional).** Con `LAG`, saca los tres días en que la demanda más subió respecto al día anterior. ¿Qué pasó en el primero?

In [18]:
# Tu código aquí


# Parte C · Excel o Power BI

Esta parte no se hace en el cuaderno. Abre `datos/electricidad_para_panel.xlsx` en Excel o cárgalo en Power BI.

**C1.** Haz una tabla dinámica (en Power BI, una matriz) con la demanda media: los meses en filas y los años en columnas. Hay una celda que no se puede comparar con las demás. ¿Cuál es y por qué?

**C2.** Haz un gráfico de la generación de 2025 por tecnología (hoja `generacion`), ordenado de mayor a menor. Ponle un título que diga la conclusión, no solo el tema.

Entrega una captura de cada una. Puedes pegarlas en un documento o guardarlas como imagen.

# Parte D · Explica (obligatoria)

Contesta con tus palabras. Frases cortas.

**D1.** Elige una decisión que hayas tenido que tomar en esta prueba y explica por qué la tomaste (4 a 6 líneas).

*(tu respuesta)*

**D2.** Cuenta tu resultado más interesante a alguien que no sabe de datos (3 líneas).

*(tu respuesta)*

**D3.** Di un límite de tu análisis, o algo que comprobarías si tuvieras más tiempo.

*(tu respuesta)*

**D4.** Si has usado una IA, di para qué y cómo comprobaste que el resultado era correcto. Si no la has usado, dilo.

*(tu respuesta)*